# Tiny GPT — Pretraining from Scratch on Shakespeare

A ~10M parameter GPT-style transformer, implemented from scratch in PyTorch, pretrained on the Tiny Shakespeare corpus.

**Setup:** Runtime -> Change runtime type -> select **T4 GPU**, then run the cells below in order.

In [ ]:
# from google.colab import files
files.download('checkpoints/ckpt.pt')

In [5]:
"""
A small GPT-style decoder-only transformer, implemented from scratch.

Architecture: token + positional embeddings -> N transformer blocks
(causal self-attention + MLP, pre-norm, residual connections) -> final
layernorm -> linear head (weight-tied to the token embedding).

Loosely follows the GPT-2 architecture, in the spirit of Karpathy's nanoGPT,
but written independently here for clarity.
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F


@dataclass
class GPTConfig:
    vocab_size: int = 65
    block_size: int = 256      # max context length
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    dropout: float = 0.2
    bias: bool = False


class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention, implemented from scratch (no
    reliance on a pre-built attention module)."""

    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head

        self.qkv_proj = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.out_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)

        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        # causal mask, cached as a buffer (not a learned parameter)
        mask = torch.tril(torch.ones(config.block_size, config.block_size))
        self.register_buffer("mask", mask.view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.shape  # batch, sequence length, embedding dim

        qkv = self.qkv_proj(x)  # (B, T, 3*C)
        q, k, v = qkv.split(self.n_embd, dim=2)

        # reshape into heads: (B, n_head, T, head_dim)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # scaled dot-product attention, computed explicitly
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        out = att @ v  # (B, n_head, T, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.out_proj(out))
        return out


class MLP(nn.Module):
    """Position-wise feed-forward network (4x expansion, GELU)."""

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.fc_in = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.fc_out = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.fc_in(x)
        x = self.gelu(x)
        x = self.fc_out(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):
    """One transformer block: pre-norm attention + pre-norm MLP, both with
    residual connections."""

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        self.token_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Embedding(config.block_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # weight tying: share the token embedding and output projection
        self.token_emb.weight = self.lm_head.weight

        self.apply(self._init_weights)
        # scaled init for residual projections (GPT-2 style)
        for name, p in self.named_parameters():
            if name.endswith("out_proj.weight") or name.endswith("fc_out.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def num_params(self):
        n = sum(p.numel() for p in self.parameters())
        # don't double count tied embedding/head weights
        n -= self.token_emb.weight.numel()
        return n

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.config.block_size, "sequence longer than block_size"

        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x = self.token_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)

        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """Autoregressively sample new tokens given a conditioning sequence."""
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")

            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return idx


In [6]:
"""
Prepares the Tiny Shakespeare dataset for training.

This builds a custom character-level tokenizer from scratch (no external
tokenizer library) and encodes the corpus into train/val binary files.
"""
import os
import pickle
import urllib.request

import numpy as np

DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
OUT_DIR = "data"
INPUT_PATH = os.path.join(OUT_DIR, "input.txt")


def download_data():
    os.makedirs(OUT_DIR, exist_ok=True)
    if os.path.exists(INPUT_PATH):
        print(f"Found existing corpus at {INPUT_PATH}")
        return
    print(f"Downloading corpus from {DATA_URL} ...")
    urllib.request.urlretrieve(DATA_URL, INPUT_PATH)
    print("Download complete.")


def build_tokenizer(text):
    """Custom character-level tokenizer: builds a stoi/itos vocab from the
    unique characters seen in the corpus. This is the simplest possible
    tokenizer, but it is genuinely 'from scratch' and works well for a
    small, focused corpus like Shakespeare."""
    chars = sorted(list(set(text)))
    vocab_size = len(chars)
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for i, ch in enumerate(chars)}
    return stoi, itos, vocab_size


def encode(text, stoi):
    return [stoi[c] for c in text]


def main():
    download_data()

    with open(INPUT_PATH, "r", encoding="utf-8") as f:
        data = f.read()
    print(f"Corpus length: {len(data):,} characters")

    stoi, itos, vocab_size = build_tokenizer(data)
    print(f"Vocab size: {vocab_size} unique characters")

    # 90/10 train/val split
    split_idx = int(len(data) * 0.9)
    train_data = data[:split_idx]
    val_data = data[split_idx:]

    train_ids = encode(train_data, stoi)
    val_ids = encode(val_data, stoi)
    print(f"Train tokens: {len(train_ids):,} | Val tokens: {len(val_ids):,}")

    train_ids = np.array(train_ids, dtype=np.uint16)
    val_ids = np.array(val_ids, dtype=np.uint16)
    train_ids.tofile(os.path.join(OUT_DIR, "train.bin"))
    val_ids.tofile(os.path.join(OUT_DIR, "val.bin"))

    meta = {"vocab_size": vocab_size, "stoi": stoi, "itos": itos}
    with open(os.path.join(OUT_DIR, "meta.pkl"), "wb") as f:
        pickle.dump(meta, f)

    print("Saved train.bin, val.bin, and meta.pkl")


if __name__ == "__main__":
    main()

Download complete.
Corpus length: 1,115,394 characters
Vocab size: 65 unique characters
Train tokens: 1,003,854 | Val tokens: 111,540
Saved train.bin, val.bin, and meta.pkl


In [ ]:
"""
Pretraining loop for the tiny GPT model.

On a Colab T4 GPU this trains a ~10M parameter model on Tiny Shakespeare
in roughly 15-25 minutes and reaches a validation loss around 1.4-1.5
(character-level), which produces fairly coherent Shakespeare-flavored
text.
"""
import math
import os
import pickle
import time

import numpy as np
import torch

# Ensure model configuration is accessible
# (Using the GPT and GPTConfig classes defined in the earlier cell)

# ---------------------------------------------------------------------------
# Config — tuned for a ~10M param model on a single Colab T4 GPU
# ---------------------------------------------------------------------------
out_dir = "checkpoints"
data_dir = "data"

block_size = 256
batch_size = 64
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

max_iters = 5000
eval_interval = 250
eval_iters = 200
log_interval = 50

learning_rate = 3e-4
min_lr = 3e-5
warmup_iters = 200
lr_decay_iters = max_iters
weight_decay = 0.1
grad_clip = 1.0

device = "cuda" if torch.cuda.is_available() else "cpu"
seed = 1337
# ---------------------------------------------------------------------------

torch.manual_seed(seed)
os.makedirs(out_dir, exist_ok=True)


def load_data():
    train_path = os.path.join(data_dir, "train.bin")
    val_path = os.path.join(data_dir, "val.bin")
    if not (os.path.exists(train_path) and os.path.exists(val_path)):
        raise FileNotFoundError(
            f"train.bin or val.bin not found at {data_dir}. Please run the data preparation cell first."
        )
    train_data = np.memmap(train_path, dtype=np.uint16, mode="r")
    val_data = np.memmap(val_path, dtype=np.uint16, mode="r")
    return train_data, val_data


def get_batch(split, train_data, val_data):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i + block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1:i + 1 + block_size].astype(np.int64)) for i in ix])
    if device == "cuda":
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y


@torch.no_grad()
def estimate_loss(model, train_data, val_data):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, train_data, val_data)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def get_lr(it):
    # linear warmup
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    # cosine decay to min_lr
    if it > lr_decay_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)


def main():
    train_data, val_data = load_data()

    with open(os.path.join(data_dir, "meta.pkl"), "rb") as f:
        meta = pickle.load(f)
    vocab_size = meta["vocab_size"]

    config = GPTConfig(
        vocab_size=vocab_size,
        block_size=block_size,
        n_layer=n_layer,
        n_head=n_head,
        n_embd=n_embd,
        dropout=dropout,
    )
    model = GPT(config).to(device)
    print(f"Model params: {model.num_params():,}")
    print(f"Device: {device}")

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=learning_rate, weight_decay=weight_decay, betas=(0.9, 0.99)
    )

    best_val_loss = float("inf")
    t0 = time.time()

    for it in range(max_iters + 1):
        lr = get_lr(it)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        if it % eval_interval == 0 or it == max_iters:
            losses = estimate_loss(model, train_data, val_data)
            print(f"step {it}: train loss {losses["train"]:.4f}, val loss {losses["val"]:.4f}")
            if losses["val"] < best_val_loss:
                best_val_loss = losses["val"]
                torch.save(
                    {
                        "model_state_dict": model.state_dict(),
                        "config": config,
                        "iter": it,
                        "val_loss": best_val_loss,
                    },
                    os.path.join(out_dir, "ckpt.pt"),
                )
                print(f"  -> saved checkpoint (val loss {best_val_loss:.4f})")

        X, Y = get_batch("train", train_data, val_data)
        _, loss = model(X, Y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        if it % log_interval == 0:
            dt = time.time() - t0
            print(f"iter {it}: loss {loss.item():.4f}, lr {lr:.2e}, {dt:.1f}s elapsed")

    print(f"Training complete. Best val loss: {best_val_loss:.4f}")


if __name__ == "__main__":
    main()

Model params: 10,725,120
Device: cuda
step 0: train loss 4.2873, val loss 4.2822
  -> saved checkpoint (val loss 4.2822)
iter 0: loss 4.2643, lr 1.50e-06, 72.3s elapsed
iter 50: loss 2.6802, lr 7.65e-05, 97.6s elapsed
iter 100: loss 2.5185, lr 1.51e-04, 123.6s elapsed
iter 150: loss 2.5055, lr 2.26e-04, 149.6s elapsed
iter 200: loss 2.4311, lr 3.00e-04, 175.5s elapsed
step 250: train loss 2.3054, val loss 2.3292
  -> saved checkpoint (val loss 2.3292)
iter 250: loss 2.3558, lr 3.00e-04, 274.2s elapsed
iter 300: loss 2.2729, lr 3.00e-04, 299.4s elapsed
iter 350: loss 2.2072, lr 2.99e-04, 325.3s elapsed
iter 400: loss 2.1094, lr 2.99e-04, 351.3s elapsed
iter 450: loss 2.0143, lr 2.98e-04, 377.2s elapsed
step 500: train loss 1.8405, val loss 1.9696
  -> saved checkpoint (val loss 1.9696)
iter 500: loss 1.9133, lr 2.97e-04, 474.9s elapsed
iter 550: loss 1.8752, lr 2.96e-04, 499.9s elapsed
iter 600: loss 1.8319, lr 2.95e-04, 525.6s elapsed
iter 650: loss 1.7863, lr 2.94e-04, 551.4s elapsed


In [10]:
"""
Interactive Testing Tab / Interface for Trained GPT Model
"""
import os
import pickle
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output

# Configuration paths
ckpt_path = "checkpoints/ckpt.pt"
meta_path = "data/meta.pkl"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = None
encode = None
decode = None

def load_model_for_testing():
    global model, encode, decode
    if not os.path.exists(ckpt_path):
        print(f"[Error] Checkpoint not found at '{ckpt_path}'. Please run the training cell first.")
        return False
    if not os.path.exists(meta_path):
        print(f"[Error] Metadata file not found at '{meta_path}'. Please run the data prep cell first.")
        return False

    # Load vocab mapping
    with open(meta_path, "rb") as f:
        meta = pickle.load(f)
    stoi, itos = meta["stoi"], meta["itos"]
    encode = lambda s: [stoi[c] for c in s if c in stoi]
    decode = lambda l: "".join([itos[i] for i in l])

    # Load model configuration & weights safely
    torch.serialization.add_safe_globals([GPTConfig])
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=True)
    config = checkpoint["config"]

    model = GPT(config)
    state_dict = checkpoint["model_state_dict"]
    unwanted_prefix = "_orig_mod."
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    print("Successfully loaded model checkpoint!")
    return True

# UI Setup
prompt_input = widgets.Textarea(
    value='ROMEO:\nShall I speak more, or shall I hear at this?',
    placeholder='Type your test prompt here...',
    description='Prompt:',
    layout=widgets.Layout(width='90%', height='100px')
)

tokens_slider = widgets.IntSlider(value=200, min=10, max=1000, step=10, description='Max Tokens')
temp_slider = widgets.FloatSlider(value=0.8, min=0.1, max=1.5, step=0.1, description='Temp')
topk_slider = widgets.IntSlider(value=50, min=1, max=100, step=1, description='Top K')

generate_button = widgets.Button(description="Generate", button_style='primary')
output_area = widgets.Output()

def on_generate_clicked(b):
    with output_area:
        clear_output()
        if model is None:
            success = load_model_for_testing()
            if not success:
                return

        print("Generating...\n")
        user_prompt = prompt_input.value
        start_ids = encode(user_prompt)
        if not start_ids:
            start_ids = [0]

        x = torch.tensor(start_ids, dtype=torch.long, device=device).unsqueeze(0)

        with torch.no_grad():
            y = model.generate(
                x,
                max_new_tokens=tokens_slider.value,
                temperature=temp_slider.value,
                top_k=topk_slider.value
            )
            result_text = decode(y[0].tolist())

        print("=" * 60)
        print(result_text)
        print("=" * 60)

generate_button.on_click(on_generate_clicked)

# Quick load attempt
load_model_for_testing()

# Display Tab Interface
controls = widgets.VBox([prompt_input, tokens_slider, temp_slider, topk_slider, generate_button])
tab = widgets.Tab()
tab.children = [widgets.VBox([controls, output_area])]
tab.set_title(0, 'Inference Playground')
display(tab)

Successfully loaded model checkpoint!
